In [1]:
print("Here begins the BN models notebook.")

Here begins the BN models notebook.


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/world_happiness_bn_clean.csv")

df_bn.head()

,Country,Climate_Index,Corruption_Perception,Crime_Rate,Education_Index,Employment_Rate,Freedom,GDP_per_Capita,Generosity,Happiness_Score,Healthy_Life_Expectancy,Income_Inequality,Internet_Access,Life_Satisfaction,Mental_Health_Index,Political_Stability,Population,Public_Health_Expenditure,Public_Trust,Social_Support,Unemployment_Rate,Urbanization_Rate,Work_Life_Balance,Year_bin
0,Australia,bin_2,bin_3,bin_2,bin_4,bin_1,bin_0,bin_1,bin_4,bin_2,bin_2,bin_3,bin_4,bin_0,bin_4,bin_3,bin_1,bin_0,bin_2,bin_3,bin_2,bin_2,bin_3,2005
1,Australia,bin_4,bin_1,bin_3,bin_1,bin_2,bin_2,bin_1,bin_0,bin_4,bin_0,bin_0,bin_2,bin_1,bin_4,bin_3,bin_2,bin_1,bin_1,bin_4,bin_0,bin_3,bin_2,2005
2,Australia,bin_3,bin_1,bin_3,bin_4,bin_2,bin_3,bin_4,bin_3,bin_3,bin_1,bin_1,bin_3,bin_4,bin_3,bin_4,bin_1,bin_2,bin_3,bin_2,bin_0,bin_4,bin_2,2005
3,Australia,bin_0,bin_1,bin_3,bin_0,bin_0,bin_4,bin_1,bin_3,bin_4,bin_3,bin_0,bin_4,bin_3,bin_0,bin_4,bin_3,bin_3,bin_3,bin_1,bin_4,bin_4,bin_0,2005
4,Australia,bin_0,bin_3,bin_0,bin_2,bin_2,bin_4,bin_2,bin_0,bin_1,bin_3,bin_4,bin_3,bin_2,bin_0,bin_2,bin_0,bin_3,bin_0,bin_3,bin_3,bin_2,bin_3,2005


In [3]:
# Inspect shape, data types, and missing values
print(df_bn.shape)
print(df_bn.dtypes)

print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(200, 24)
Country                        str
Climate_Index                  str
Corruption_Perception          str
Crime_Rate                     str
Education_Index                str
Employment_Rate                str
Freedom                        str
GDP_per_Capita                 str
Generosity                     str
Happiness_Score                str
Healthy_Life_Expectancy        str
Income_Inequality              str
Internet_Access                str
Life_Satisfaction              str
Mental_Health_Index            str
Political_Stability            str
Population                     str
Public_Health_Expenditure      str
Public_Trust                   str
Social_Support                 str
Unemployment_Rate              str
Urbanization_Rate              str
Work_Life_Balance              str
Year_bin                     int64
dtype: object

Missing values per column:
Country                      0
Climate_Index                0
Corruption_Perception        0
Crime_Rate     

In [4]:
# Convert all columns to string so pgmpy treats them as discrete states
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Country                      string
Climate_Index                string
Corruption_Perception        string
Crime_Rate                   string
Education_Index              string
Employment_Rate              string
Freedom                      string
GDP_per_Capita               string
Generosity                   string
Happiness_Score              string
Healthy_Life_Expectancy      string
Income_Inequality            string
Internet_Access              string
Life_Satisfaction            string
Mental_Health_Index          string
Political_Stability          string
Population                   string
Public_Health_Expenditure    string
Public_Trust                 string
Social_Support               string
Unemployment_Rate            string
Urbanization_Rate            string
Work_Life_Balance            string
Year_bin                     string
dtype: object


In [5]:
# Check number of states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn.columns,
    "Num_States": [df_bn[col].nunique(dropna=True) for col in df_bn.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
0,Country,10
1,Climate_Index,5
2,Corruption_Perception,5
3,Crime_Rate,5
4,Education_Index,5
5,Employment_Rate,5
6,Freedom,5
7,GDP_per_Capita,5
8,Generosity,5
9,Happiness_Score,5


In [6]:
# Create BN modeling dataset
df_bn_model = df_bn.copy()

# Drop Country to avoid high-cardinality country identity dominating the graph
df_bn_model = df_bn_model.drop(columns=["Country"], errors="ignore")

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(200, 23)
['Climate_Index', 'Corruption_Perception', 'Crime_Rate', 'Education_Index', 'Employment_Rate', 'Freedom', 'GDP_per_Capita', 'Generosity', 'Happiness_Score', 'Healthy_Life_Expectancy', 'Income_Inequality', 'Internet_Access', 'Life_Satisfaction', 'Mental_Health_Index', 'Political_Stability', 'Population', 'Public_Health_Expenditure', 'Public_Trust', 'Social_Support', 'Unemployment_Rate', 'Urbanization_Rate', 'Work_Life_Balance', 'Year_bin']


In [7]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pgmpy.estimators import HillClimbSearch, ExpertKnowledge, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

In [9]:
# Learn an unconstrained Bayesian Network structure
hc = HillClimbSearch(df_bn_model)

learned_dag = hc.estimate(
    scoring_method="k2",
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag.nodes()))
print("Number of edges:", len(learned_dag.edges()))
print("\nLearned edges:")
print(sorted(learned_dag.edges()))

Number of nodes: 23
Number of edges: 9

Learned edges:
[('Climate_Index', 'Income_Inequality'), ('Crime_Rate', 'GDP_per_Capita'), ('Education_Index', 'GDP_per_Capita'), ('Internet_Access', 'Income_Inequality'), ('Social_Support', 'Education_Index'), ('Social_Support', 'GDP_per_Capita'), ('Social_Support', 'Income_Inequality'), ('Unemployment_Rate', 'Education_Index'), ('Year_bin', 'Education_Index')]


In [17]:
# Convert the learned graph into a discrete Bayesian Network
# Important: add all columns as nodes, including isolated ones
bn_unconstrained = DiscreteBayesianNetwork()
bn_unconstrained.add_nodes_from(df_bn_model.columns)
bn_unconstrained.add_edges_from(learned_dag.edges())

# Fit CPDs using Bayesian parameter estimation
bn_unconstrained.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Model check:", bn_unconstrained.check_model())

Model check: True


In [18]:
# Inspect direct parents of Life_Satisfaction
if "Life_Satisfaction" in bn_unconstrained.nodes():
    print("Parents of Life_Satisfaction in unconstrained BN:")
    print(list(bn_unconstrained.predecessors("Life_Satisfaction")))

# Inspect CPD metadata for the target variable
if "Life_Satisfaction" in bn_unconstrained.nodes():
    cpd_ls = bn_unconstrained.get_cpds("Life_Satisfaction")
    print("\nCPD state names for Life_Satisfaction:")
    print(cpd_ls.state_names)


Parents of Life_Satisfaction in unconstrained BN:
[]

CPD state names for Life_Satisfaction:
{'Life_Satisfaction': ['bin_0', 'bin_1', 'bin_2', 'bin_3', 'bin_4']}


In [20]:
# Build a restricted BN with expert constraints
# The goal is to prevent implausible directions and encourage a more interpretable structure
# for Life_Satisfaction in this macro-level dataset.

forbidden_edges = [
    # Life satisfaction should not cause macro indicators
    ("Life_Satisfaction", "GDP_per_Capita"),
    ("Life_Satisfaction", "Social_Support"),
    ("Life_Satisfaction", "Healthy_Life_Expectancy"),
    ("Life_Satisfaction", "Freedom"),
    ("Life_Satisfaction", "Generosity"),
    ("Life_Satisfaction", "Corruption_Perception"),
    ("Life_Satisfaction", "Unemployment_Rate"),
    ("Life_Satisfaction", "Education_Index"),
    ("Life_Satisfaction", "Population"),
    ("Life_Satisfaction", "Urbanization_Rate"),
    ("Life_Satisfaction", "Public_Trust"),
    ("Life_Satisfaction", "Mental_Health_Index"),
    ("Life_Satisfaction", "Income_Inequality"),
    ("Life_Satisfaction", "Public_Health_Expenditure"),
    ("Life_Satisfaction", "Climate_Index"),
    ("Life_Satisfaction", "Work_Life_Balance"),
    ("Life_Satisfaction", "Internet_Access"),
    ("Life_Satisfaction", "Crime_Rate"),
    ("Life_Satisfaction", "Political_Stability"),
    ("Life_Satisfaction", "Employment_Rate"),
    ("Life_Satisfaction", "Happiness_Score"),
    ("Life_Satisfaction", "Year_bin"),

    # Time should not be caused by current indicators
    ("GDP_per_Capita", "Year_bin"),
    ("Social_Support", "Year_bin"),
    ("Healthy_Life_Expectancy", "Year_bin"),
    ("Freedom", "Year_bin"),
    ("Generosity", "Year_bin"),
    ("Corruption_Perception", "Year_bin"),
    ("Unemployment_Rate", "Year_bin"),
    ("Education_Index", "Year_bin"),
    ("Population", "Year_bin"),
    ("Urbanization_Rate", "Year_bin"),
    ("Public_Trust", "Year_bin"),
    ("Mental_Health_Index", "Year_bin"),
    ("Income_Inequality", "Year_bin"),
    ("Public_Health_Expenditure", "Year_bin"),
    ("Climate_Index", "Year_bin"),
    ("Work_Life_Balance", "Year_bin"),
    ("Internet_Access", "Year_bin"),
    ("Crime_Rate", "Year_bin"),
    ("Political_Stability", "Year_bin"),
    ("Employment_Rate", "Year_bin"),
    ("Happiness_Score", "Year_bin"),
    ("Life_Satisfaction", "Year_bin"),

    # Optional: Happiness score should not cause the underlying macro drivers
    ("Happiness_Score", "GDP_per_Capita"),
    ("Happiness_Score", "Social_Support"),
    ("Happiness_Score", "Healthy_Life_Expectancy"),
    ("Happiness_Score", "Freedom"),
    ("Happiness_Score", "Public_Trust"),
    ("Happiness_Score", "Mental_Health_Index"),
]

In [21]:
expert_knowledge = ExpertKnowledge(
    forbidden_edges=forbidden_edges
)

hc_restricted = HillClimbSearch(df_bn_model)

learned_dag_restricted = hc_restricted.estimate(
    scoring_method="k2",
    expert_knowledge=expert_knowledge,
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag_restricted.nodes()))
print("Number of edges:", len(learned_dag_restricted.edges()))
print("\nRestricted BN edges:")
print(sorted(learned_dag_restricted.edges()))

Number of nodes: 23
Number of edges: 9

Restricted BN edges:
[('Climate_Index', 'Income_Inequality'), ('Crime_Rate', 'GDP_per_Capita'), ('Education_Index', 'GDP_per_Capita'), ('Internet_Access', 'Income_Inequality'), ('Social_Support', 'Education_Index'), ('Social_Support', 'GDP_per_Capita'), ('Social_Support', 'Income_Inequality'), ('Unemployment_Rate', 'Education_Index'), ('Year_bin', 'Education_Index')]


In [22]:
# Fit the restricted BN
# Important: add all columns as nodes so isolated variables are preserved
bn_restricted = DiscreteBayesianNetwork()
bn_restricted.add_nodes_from(df_bn_model.columns)
bn_restricted.add_edges_from(learned_dag_restricted.edges())

bn_restricted.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Restricted model check:", bn_restricted.check_model())

Restricted model check: True


In [23]:
# Inspect direct parents of Life_Satisfaction in the restricted BN
if "Life_Satisfaction" in bn_restricted.nodes():
    print("Parents of Life_Satisfaction in restricted BN:")
    print(list(bn_restricted.predecessors("Life_Satisfaction")))

    print("\nChildren of Life_Satisfaction in restricted BN:")
    print(list(bn_restricted.successors("Life_Satisfaction")))

Parents of Life_Satisfaction in restricted BN:
[]

Children of Life_Satisfaction in restricted BN:
[]
